In [1]:
from pathlib import Path

import json
import time
import joblib
import numpy as np
import pandas as pd

current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

processed_data_path = (
    project_root
    / "data"
    / "processed"
    / "heart_disease_clean.csv"
)

comparison_report_path = (
    project_root
    / "reports"
    / "model_comparison.csv"
)

if not processed_data_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {processed_data_path}"
    )

if not comparison_report_path.exists():
    raise FileNotFoundError(
        f"Model comparison report not found: "
        f"{comparison_report_path}"
    )

df = pd.read_csv(processed_data_path)
comparison_results = pd.read_csv(
    comparison_report_path
)

target_column = "heart_disease"

if target_column not in df.columns:
    raise KeyError(
        f"Target column '{target_column}' was not found."
    )

required_result_columns = [
    "Model",
    "PR-AUC",
    "ROC-AUC",
    "Recall",
    "F1 Score"
]

missing_result_columns = [
    column
    for column in required_result_columns
    if column not in comparison_results.columns
]

if missing_result_columns:
    raise KeyError(
        f"Missing model-comparison columns: "
        f"{missing_result_columns}"
    )

if "Dataset" in comparison_results.columns:
    comparison_results = comparison_results[
        comparison_results["Dataset"] == "Testing"
    ].copy()

ranked_results = (
    comparison_results
    .sort_values(
        by=[
            "PR-AUC",
            "ROC-AUC",
            "Recall",
            "F1 Score"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

best_model_name = ranked_results.loc[0, "Model"]
best_model_result = ranked_results.iloc[0]

X = df.drop(columns=[target_column])
y = df[target_column].astype("int64")

print("Dataset and comparison report loaded successfully")
print("Dataset shape:", df.shape)
print("Feature matrix shape:", X.shape)
print("Selected model:", best_model_name)
print(
    "Selected model PR-AUC:",
    round(float(best_model_result["PR-AUC"]), 4)
)
print(
    "Selected model ROC-AUC:",
    round(float(best_model_result["ROC-AUC"]), 4)
)

Dataset and comparison report loaded successfully
Dataset shape: (4238, 16)
Feature matrix shape: (4238, 15)
Selected model: Logistic Regression
Selected model PR-AUC: 0.2937
Selected model ROC-AUC: 0.6952
